# Overlap between the DILI gold-standard dataset and the three in vitro assay datasets

This checks a specific risk for General Response G7 (linking the in vitro assay models to the DILI models): the plan is to use the predictions of the consensus MMP and caspase-3/7 models as extra input features for a DILI model. If a compound in the DILI held-out test set (Held-223) was also used to train the MMP/caspase models, then that compound's MMP/caspase-predicted probability is not a genuine held-out prediction for it -- the MMP/caspase model has already seen it during training. Using that feature for the DILI model would leak information.

Matching convention used throughout the rest of this project, overlap is checked by InChIKey (computed from canonicalised SMILES with RDKit), not by raw SMILES text, since the same compound can be written in more than one valid SMILES form. Each assay dataset is restricted to the Active/Inactive compounds actually used for modelling (Inconclusive compounds are excluded during training, per the Methods), so the counts here match the real training data, not the raw cleaned file.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

def canonicalize(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        return Chem.MolToSmiles(mol) if mol else None
    except Exception:
        return None

def smiles_to_inchikey(smi):
    csmi = canonicalize(smi)
    if csmi is None:
        return None
    mol = Chem.MolFromSmiles(csmi)
    try:
        return inchi.MolToInchiKey(mol) or None
    except Exception:
        return None

## 1. Load the three in vitro assay datasets (restricted to Active/Inactive, matching what is actually used for modelling) and the three DILI files

In [2]:
ASSAY_DATASETS = {
    'MMP (AID 720637)': ('Summary_1_models/cleaning/AID_720637_cleaned.csv', 'PUBCHEM_EXT_DATASOURCE_SMILES'),
    'Caspase HepG2 (654+1347034)': ('summary_2_1437034/modeling/wahab_jobs/cleaned_data/AID_654_1347034_cleaned.csv', 'PUBCHEM_EXT_DATASOURCE_SMILES'),
    'Caspase CHO-K1 (1347037)': ('summary_1347037/modeling/wahab_jobs/cleaned_data/AID_1347037_cleaned.csv', 'PUBCHEM_EXT_DATASOURCE_SMILES'),
}
DILI_DATASETS = {
    'Held-223 (DILI test)': '../liver/test_data_heldouttest_DILIst_223.csv',
    'NCV-888 (DILI train)': '../liver/test_data_ncv_DILIst_888.csv',
    'Gold-1111 (full)': '../liver/DILI_Goldstandard_1111.csv',
}

assay_ikeys = {}
for name, (path, col) in ASSAY_DATASETS.items():
    df = pd.read_csv(path, usecols=['PUBCHEM_CID', col, 'PUBCHEM_ACTIVITY_OUTCOME'])
    df = df[df['PUBCHEM_ACTIVITY_OUTCOME'].isin(['Active', 'Inactive'])]  # matches actual training data
    assay_ikeys[name] = set(df[col].apply(smiles_to_inchikey).dropna())
    print(f'{name}: {len(df)} compounds used for modelling, {len(assay_ikeys[name])} unique InChIKeys')

print()
dili_ikeys = {}
for name, path in DILI_DATASETS.items():
    df = pd.read_csv(path, usecols=['smiles_r'])
    dili_ikeys[name] = set(df['smiles_r'].apply(smiles_to_inchikey).dropna())
    print(f'{name}: {len(dili_ikeys[name])} unique InChIKeys')

MMP (AID 720637): 5937 compounds used for modelling, 5937 unique InChIKeys
Caspase HepG2 (654+1347034): 7285 compounds used for modelling, 7283 unique InChIKeys
Caspase CHO-K1 (1347037): 7496 compounds used for modelling, 7496 unique InChIKeys

Held-223 (DILI test): 222 unique InChIKeys
NCV-888 (DILI train): 874 unique InChIKeys
Gold-1111 (full): 1096 unique InChIKeys


## 2. Overlap, by InChIKey, between each DILI file and each assay dataset

In [3]:
overlap_table = []
for dname, dikeys in dili_ikeys.items():
    print(f'--- {dname} (n={len(dikeys)} unique) ---')
    for aname, aikeys in assay_ikeys.items():
        overlap = dikeys & aikeys
        pct = 100 * len(overlap) / len(dikeys)
        print(f'  overlap with {aname}: {len(overlap)} ({pct:.1f}%)')
        overlap_table.append({'DILI set': dname, 'Assay': aname, 'DILI n': len(dikeys),
                               'Overlap n': len(overlap), 'Overlap %': round(pct, 1)})
    print()

overlap_df = pd.DataFrame(overlap_table)

--- Held-223 (DILI test) (n=222 unique) ---
  overlap with MMP (AID 720637): 62 (27.9%)
  overlap with Caspase HepG2 (654+1347034): 79 (35.6%)
  overlap with Caspase CHO-K1 (1347037): 80 (36.0%)

--- NCV-888 (DILI train) (n=874 unique) ---
  overlap with MMP (AID 720637): 168 (19.2%)
  overlap with Caspase HepG2 (654+1347034): 228 (26.1%)
  overlap with Caspase CHO-K1 (1347037): 236 (27.0%)

--- Gold-1111 (full) (n=1096 unique) ---
  overlap with MMP (AID 720637): 230 (21.0%)
  overlap with Caspase HepG2 (654+1347034): 307 (28.0%)
  overlap with Caspase CHO-K1 (1347037): 316 (28.8%)


## 3. Save results

In [4]:
import os
os.makedirs('results', exist_ok=True)
overlap_df.to_csv('results/dili_assay_overlap.csv', index=False)
print('Saved -> results/dili_assay_overlap.csv')

Saved -> results/dili_assay_overlap.csv


## Conclusion

The overlap is substantial and consistent across all three assay datasets, for both the DILI training set and the held-out test set. Most importantly for G7: **27.9% of the DILI held-out test set (62 of 222 compounds) is also in the MMP training data, and 35.6-36.0% (79-80 of 222) is also in each caspase-3/7 training dataset**.

This means the plan in G7 -- using the consensus MMP/caspase-3/7 models' predicted probabilities as extra input features for a DILI model -- cannot be built the straightforward way (train the MMP/caspase models once on all their data, then score every DILI compound with them) without leaking test-set information for a large fraction of the DILI held-out test set. For those compounds, the MMP/caspase model has already seen the compound during its own training, so its prediction for that compound is not a genuine out-of-sample estimate.

This needs to be resolved before G7's experiment is built, most likely by generating the MMP/caspase proxy features out-of-fold (e.g. using the existing 10-fold cross-validation splits, so each compound's proxy feature comes only from folds that did not train on it) rather than from a single model trained on the full assay dataset.